In [1]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
import chromadb
#from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

#load_dotenv()

c:\Users\Kamal\OneDrive\Documents\Kamal_files\GitHub_Folders\genai-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="manufacturing-sample-genai-v2.pdf",
    strategy="hi_res",        # best extraction quality
    infer_table_structure=True,
    extract_images_in_pdf=False
)

# Merge all elements into one clean text
full_text = "\n".join([el.text for el in elements if el.text])

print(len(full_text))
print(full_text[:1000])

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 4105.22it/s]


44897
1. Introduction
This document describes the manufacturing process, materials, and technical specifications for the production of industrial lithium-ion battery modules used in electric vehicles and energy storage systems. The goal of this document is to provide suppliers, manufacturing partners, and engineering teams with a clear understanding of the production pipeline, component selection, and quality assurance requirements involved in producing high-performance battery modules.
The battery modules are designed to deliver high energy density, long cycle life, and safe operation under varying environmental conditions. These modules are intended for applications such as electric mobility platforms, renewable energy storage systems, industrial backup power units, and grid stabilization infrastructure. As demand for electrification and sustainable energy solutions continues to grow globally, lithium-ion battery technology plays a critical role in enabling efficient and reliable ene

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_text(full_text)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 67
1. Introduction
This document describes the manufacturing process, materials, and technical specifications for the production of industrial lithium-ion battery modules used in electric vehicles and energy storage systems. The goal of this document is to provide suppliers, manufacturing partners, and engineering teams with a clear understanding of the production pipeline, component selection, and quality assurance requirements involved in producing high-performance battery modules.


In [4]:
documents = []

for i, chunk in enumerate(chunks):
    documents.append({
        "text": chunk,
        "metadata": {
            "chunk_id": i,
            "source": "document.pdf"
        }
    })

print(documents[1])

{'text': 'The battery modules are designed to deliver high energy density, long cycle life, and safe operation under varying environmental conditions. These modules are intended for applications such as electric mobility platforms, renewable energy storage systems, industrial backup power units, and grid stabilization infrastructure. As demand for electrification and sustainable energy solutions continues to grow globally, lithium-ion battery technology plays a critical role in enabling efficient and reliable energy storage.', 'metadata': {'chunk_id': 1, 'source': 'document.pdf'}}


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode([doc["text"] for doc in documents])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6215.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
client = chromadb.Client()
collection = client.create_collection("rag_docs")

for i, doc in enumerate(documents):
    collection.add(
        documents=[doc["text"]],
        embeddings=[embeddings[i]],
        metadatas=[doc["metadata"]],
        ids=[str(i)]
    )

In [10]:
query = "what is the Operating Voltage Range?"

query_embedding = model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

docs = results["documents"][0]
metas = results["metadatas"][0]
distances = results["distances"][0]

for i, (doc, meta, dist) in enumerate(zip(docs, metas, distances), 1):
    similarity = dist   # convert distance → similarity
    
    print(f"\nResult {i}")
    print("-"*60)
    print("Similarity:", round(similarity, 3))
    print("Metadata:", meta)
    print("Text:", doc, "...")


Result 1
------------------------------------------------------------
Similarity: 0.884
Metadata: {'chunk_id': 32, 'source': 'document.pdf'}
Text: These parameters define the safe operating limits and expected performance behavior of the module under standard operating conditions. System integrators should ensure
that the battery module is operated within these limits to maximize performance, maintain safety, and prolong battery lifespan.
Operating Voltage Range
40V – 54.6V
The EV-PowerCell module operates within a voltage range of 40 volts to 54.6 volts. The lower limit represents the recommended discharge cutoff voltage to prevent deep discharge of the battery cells, which could cause long-term degradation of the electrode materials. ...

Result 2
------------------------------------------------------------
Similarity: 1.013
Metadata: {'source': 'document.pdf', 'chunk_id': 33}
Text: The upper voltage limit corresponds to the fully charged state of the battery module. Charging beyond